In [ ]:
# ==========================================
# 0. IMPORT LIBRARIES
# ==========================================

import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
import os
import pickle

# ==========================================
# 1. LOAD DATA
# ==========================================

file_path = "./rna_malignant_counts.h5ad"

print(f"Loading {file_path}...")
adata = sc.read_h5ad(file_path)
print(f"Initial Shape: {adata.shape}")

In [ ]:
# ==========================================
# 2. CREATE CONDITION / CELL TYPE METADATA
# ==========================================

condition_map = {
    "pre-IR": "ctrl",
    "24h post-IR": "IR_24h",
    "1 week post-IR": "IR_1week",
}

if "Timepoint" not in adata.obs:
    raise KeyError("CRITICAL: 'Timepoint' column missing in raw data!")

adata.obs["condition"] = adata.obs["Timepoint"].map(condition_map)

if adata.obs["condition"].isna().any():
    print("Unmapped Timepoint values:")
    print(adata.obs.loc[adata.obs["condition"].isna(), "Timepoint"].unique())
    raise ValueError("Some Timepoint values were not mapped to condition labels.")

adata.obs["condition"] = adata.obs["condition"].astype("category")

# Some scGPT/GEARS utilities expect this
adata.obs["condition_name"] = adata.obs["condition"].astype(str).astype("category")

if "GBM_malig_subtypes" not in adata.obs:
    raise KeyError("CRITICAL: 'GBM_malig_subtypes' column missing in raw data!")

adata.obs["cell_type"] = (
    adata.obs["GBM_malig_subtypes"]
    .astype(str)
    .str.replace("-", "_", regex=False)
    .astype("category")
)

print("Conditions found:")
print(adata.obs["condition"].value_counts())

print("\nCell types found:")
print(adata.obs["cell_type"].value_counts())

In [ ]:
# ==========================================
# 3. GENE MAPPING, NORMALIZATION & FILTERING
# ==========================================

orthology_file = "./mouse2human.pkl"
print("\nMapping mouse genes to human orthologs...")

try:
    with open(orthology_file, "rb") as f:
        mouse2human = pickle.load(f)
    print(f"Loaded mapping for {len(mouse2human)} genes.")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Missing {orthology_file}. Please ensure it is in the folder."
    )

# Keep only mouse genes with human orthologs
valid_mouse_genes = [g for g in adata.var_names if g in mouse2human]
print(
    f"Filtering: {len(adata.var_names)} original genes -> "
    f"{len(valid_mouse_genes)} mapped genes."
)

adata = adata[:, valid_mouse_genes].copy()

# Add human gene names
adata.var["human_gene"] = [mouse2human[g] for g in adata.var_names]

# Collapse duplicated human orthologs
# This is better than var_names_make_unique()
# print("Collapsing duplicated human orthologs...")
#
# if sparse.issparse(adata.X):
#     expression_df = pd.DataFrame.sparse.from_spmatrix(
#         adata.X,
#         index=adata.obs_names,
#         columns=adata.var["human_gene"],
#     )
# else:
#     expression_df = pd.DataFrame(
#         adata.X,
#         index=adata.obs_names,
#         columns=adata.var["human_gene"],
#     )
#
# expression_df = expression_df.groupby(axis=1, level=0).sum()
#
# adata = sc.AnnData(
#     X=expression_df.values,
#     obs=adata.obs.copy(),
#     var=pd.DataFrame(index=expression_df.columns),
# )
#
# adata.var["gene_name"] = adata.var_names.astype(str)
#
# print(f"Shape after ortholog collapsing: {adata.shape}")
# ==========================================
# FAST COLLAPSE OF DUPLICATED HUMAN ORTHOLOGS
# while preserving author's highly_variable labels
# ==========================================

print("Collapsing duplicated human orthologs using sparse matrix multiplication...")

human_genes = pd.Index(adata.var["human_gene"].astype(str))

# Preserve author's HVG annotation before rebuilding AnnData
if "highly_variable" in adata.var.columns:
    hvg_by_human = (
        adata.var
        .assign(human_gene=human_genes)
        .groupby("human_gene")["highly_variable"]
        .any()
    )
else:
    hvg_by_human = None
    print("⚠️ No 'highly_variable' column found in adata.var.")

# Convert human gene names to integer group labels
codes, unique_human_genes = pd.factorize(human_genes)

n_mouse_genes = adata.n_vars
n_human_genes = len(unique_human_genes)

collapse_matrix = sparse.csr_matrix(
    (
        np.ones(n_mouse_genes, dtype=np.float32),
        (np.arange(n_mouse_genes), codes),
    ),
    shape=(n_mouse_genes, n_human_genes),
)

if sparse.issparse(adata.X):
    X = adata.X.tocsr()
else:
    X = sparse.csr_matrix(adata.X)

X_collapsed = X @ collapse_matrix

adata = sc.AnnData(
    X=X_collapsed.tocsr(),
    obs=adata.obs.copy(),
    var=pd.DataFrame(index=pd.Index(unique_human_genes, name="gene_name")),
)

adata.var["gene_name"] = adata.var_names.astype(str)

# Restore author's HVG labels onto collapsed human genes
if hvg_by_human is not None:
    adata.var["highly_variable"] = (
        hvg_by_human
        .reindex(adata.var_names)
        .fillna(False)
        .astype(bool)
        .values
    )

print(f"Shape after ortholog collapsing: {adata.shape}")
print(f"Number of duplicated mappings collapsed: {n_mouse_genes - n_human_genes}")

if "highly_variable" in adata.var.columns:
    print(f"Author HVGs retained: {adata.var['highly_variable'].sum()}")

In [ ]:
# ==========================================
# 4. SANITIZE, FILTER, NORMALIZE
# ==========================================

print("Sanitizing expression matrix...")

# Store raw counts before normalization
adata.layers["counts"] = adata.X.copy()

# Replace NaN / infinity values
if sparse.issparse(adata.X):
    adata.X.data = np.nan_to_num(
        adata.X.data,
        nan=0.0,
        posinf=1e5,
        neginf=0.0,
    )
else:
    adata.X = np.nan_to_num(
        adata.X,
        nan=0.0,
        posinf=1e5,
        neginf=0.0,
    )

# Filter genes expressed in very few cells
sc.pp.filter_genes(adata, min_cells=3)

print("Applying log-normalization...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(f"Normalization complete. Shape: {adata.shape}")

In [ ]:
# ==========================================
# FILTER USING PRE-COMPUTED AUTHOR HVG LABELS
# ==========================================

print("Filtering using the pre-computed 'highly_variable' column...")

if "highly_variable" not in adata.var.columns:
    raise KeyError(
        "The 'highly_variable' column is missing from adata.var. "
        "Check that it was preserved during ortholog mapping."
    )

print("Before HVG filtering:", adata.shape)
print("Number of HVGs:", int(adata.var["highly_variable"].sum()))

adata = adata[:, adata.var["highly_variable"] == True].copy()

adata.var["gene_name"] = adata.var_names.astype(str)

print(f"Final data shape after applying author HVGs: {adata.shape}")

In [ ]:
# ==========================================
# 6. VALIDATE: TIMEPOINT LABELS ARE NOT GENES
# ==========================================

timepoint_tokens = {"IR_24h", "IR_1week"}
gene_set = set(adata.var["gene_name"])

overlap = timepoint_tokens.intersection(gene_set)

if overlap:
    raise ValueError(
        f"Timepoint labels should not be in adata.var['gene_name']: {overlap}"
    )
else:
    print("✅ Timepoint labels are metadata only, not genes.")

In [ ]:
# ==========================================
# 7. CALCULATE DE STATS
# ==========================================

print("\nCalculating DE stats...")

adata_norm = adata.copy()

try:
    sc.tl.rank_genes_groups(
        adata_norm,
        groupby="condition",
        reference="ctrl",
        method="wilcoxon",
        use_raw=False,
    )

    adata.uns["rank_genes_groups_cov_all"] = adata_norm.uns["rank_genes_groups"]
    print("DE stats calculated successfully.")

except Exception as e:
    print(f"ERROR in DE calculation: {e}")

In [ ]:
# ==========================================
# 8. CONVERT DE STATS TO GEARS-LIKE FORMAT
# ==========================================

if "rank_genes_groups_cov_all" not in adata.uns:
    raise KeyError(
        "Could not find 'rank_genes_groups_cov_all'. "
        "Please run the previous DE step first."
    )

raw_de = adata.uns["rank_genes_groups_cov_all"]
print("Found raw Scanpy DE stats. Converting to dictionary format...")

new_de_dict = {}

names_df = pd.DataFrame(raw_de["names"])

for group in names_df.columns:
    gene_list = names_df[group].tolist()

    if gene_list and isinstance(gene_list[0], bytes):
        gene_list = [g.decode("utf-8") for g in gene_list]

    new_de_dict[group] = gene_list
    print(f"  > Processed group '{group}': {len(gene_list)} genes")

adata.uns["rank_genes_groups_cov_all"] = new_de_dict

print("Conversion complete.")

In [ ]:
adata.obs["condition"] = adata.obs["Timepoint"].map(condition_map)
adata.obs["condition_name"] = adata.obs["condition"].astype(str).astype("category")

In [ ]:
# ==========================================
# 9. SAVE OUTPUT
# ==========================================

# The output location required by scGPT/GEARS
output_dir = "./data/radiotherapyGB"
output_file = os.path.join(output_dir, "perturb_processed.h5ad")

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

print(f"\nSaving to: {output_file}")
adata.write_h5ad(output_file)
print("✅ Processing Complete. Ready for pert_data.load()")